# Phase 2: Preprocessing
**LendingClub Loan Default Prediction**

Loads only the columns we need (via `usecols`), cleans, engineers features, encodes, and saves the processed dataset.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH   = r'C:\Users\Administrator\Desktop\DataMiningProject\accepted_2007_to_2018Q4.csv'
OUTPUT_PATH = r'C:\Users\Administrator\Desktop\DataMiningProject\processed_data.csv'

MISSING_THRESH = 0.50  # drop columns with >50% missing

## 1. Define Columns to Load
We skip all post-origination, ID, and free-text columns at load time — no need to read them at all.

In [2]:
# Columns confirmed as post-origination (from EDA cell 11)
POST_ORIG = {
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
}

# Other non-predictive columns
DROP_OTHER = {
    'id', 'member_id', 'url', 'desc', 'title', 'zip_code',
    'funded_amnt', 'funded_amnt_inv',  # near-duplicate of loan_amnt
    'policy_code', 'pymnt_plan',        # single-value columns
    'emp_title',                         # high-cardinality free text
    'disbursement_method'               # not available at origination decision time
}

# Read just the header to get all column names
header = pd.read_csv(DATA_PATH, nrows=0, low_memory=False)
all_cols = list(header.columns)
print(f"Total columns in file: {len(all_cols)}")

LOAD_COLS = [c for c in all_cols if c not in POST_ORIG and c not in DROP_OTHER]
print(f"Columns to load: {len(LOAD_COLS)}")

Total columns in file: 151
Columns to load: 102


## 2. Load Full Dataset
Using `usecols` cuts memory significantly. This may take 1–2 minutes for the full 1.68 GB file.

In [3]:
print("Loading data...")
df = pd.read_csv(DATA_PATH, usecols=LOAD_COLS, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Filter to terminal loan states only
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
print(f"After filtering to Fully Paid / Charged Off: {df.shape[0]:,} rows")
print(f"Default rate: {(df['loan_status'] == 'Charged Off').mean():.2%}")

Loading data...
Loaded: 2,260,701 rows x 102 columns
After filtering to Fully Paid / Charged Off: 1,345,310 rows
Default rate: 19.96%


## 3. Create Target Variable

In [4]:
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)
df = df.drop(columns=['loan_status'])
print(f"Target distribution:\n{df['target'].value_counts()}")

Target distribution:
target
0    1076751
1     268559
Name: count, dtype: int64


## 4. Drop High-Missing Columns
57 columns in the loaded set have >50% missing values — they bring more noise than signal.

In [5]:
missing_rate = df.isnull().mean()
high_missing = missing_rate[missing_rate > MISSING_THRESH].index.tolist()

print(f"Dropping {len(high_missing)} columns with >{MISSING_THRESH:.0%} missing:")
for c in sorted(high_missing):
    print(f"  {c}  ({missing_rate[c]:.1%} missing)")

df = df.drop(columns=high_missing)
print(f"\nRemaining columns: {df.shape[1]}")

Dropping 35 columns with >50% missing:
  all_util  (60.0% missing)
  annual_inc_joint  (98.1% missing)
  dti_joint  (98.1% missing)
  il_util  (65.4% missing)
  inq_fi  (60.0% missing)
  inq_last_12m  (60.0% missing)
  max_bal_bc  (60.0% missing)
  mths_since_last_delinq  (50.5% missing)
  mths_since_last_major_derog  (73.7% missing)
  mths_since_last_record  (83.0% missing)
  mths_since_rcnt_il  (61.1% missing)
  mths_since_recent_bc_dlq  (76.3% missing)
  mths_since_recent_revol_delinq  (66.6% missing)
  open_acc_6m  (60.0% missing)
  open_act_il  (60.0% missing)
  open_il_12m  (60.0% missing)
  open_il_24m  (60.0% missing)
  open_rv_12m  (60.0% missing)
  open_rv_24m  (60.0% missing)
  revol_bal_joint  (98.6% missing)
  sec_app_chargeoff_within_12_mths  (98.6% missing)
  sec_app_collections_12_mths_ex_med  (98.6% missing)
  sec_app_earliest_cr_line  (98.6% missing)
  sec_app_fico_range_high  (98.6% missing)
  sec_app_fico_range_low  (98.6% missing)
  sec_app_inq_last_6mths  (98.6% m

## 5. Feature Engineering

In [6]:
# --- Loan-to-income ratio ---
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'].clip(lower=1))

# --- Credit age in months ---
df['issue_d_dt']      = pd.to_datetime(df['issue_d'],        format='%b-%Y', errors='coerce')
df['earliest_cr_dt']  = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
df['credit_age_months'] = ((df['issue_d_dt'] - df['earliest_cr_dt']).dt.days / 30).clip(lower=0)

# --- FICO midpoint (average of low and high) ---
if 'fico_range_high' in df.columns:
    df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
    df = df.drop(columns=['fico_range_high'])

# Drop raw date columns — already extracted what we need
df = df.drop(columns=['issue_d', 'earliest_cr_line', 'issue_d_dt', 'earliest_cr_dt'],
             errors='ignore')

print("Engineered features added: loan_to_income, credit_age_months, fico_avg")
print(df[['loan_to_income', 'credit_age_months', 'fico_avg']].describe())

Engineered features added: loan_to_income, credit_age_months, fico_avg
       loan_to_income  credit_age_months      fico_avg
count    1.345310e+06       1.345310e+06  1.345310e+06
mean     5.014272e+00       1.979567e+02  6.981851e+02
std      3.385551e+02       9.138623e+01  3.185284e+01
min      1.714286e-04       3.650000e+01  6.270000e+02
25%      1.246903e-01       1.369000e+02  6.720000e+02
50%      2.000000e-01       1.795667e+02  6.920000e+02
75%      2.909091e-01       2.435000e+02  7.120000e+02
max      4.000000e+04       1.013567e+03  8.475000e+02


## 6. Encode Categorical Features

Strategy:
- **Ordinal**: `grade`, `sub_grade`, `emp_length`, `term` — natural ordering exists
- **One-hot**: `home_ownership`, `purpose`, `verification_status`, `addr_state`, `application_type`, `initial_list_status`

In [7]:
# --- term: strip whitespace, map to integer months ---
df['term'] = df['term'].str.strip().str.extract(r'(\d+)').astype(float)

# --- grade: A=1 ... G=7 ---
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade'] = df['grade'].map(grade_map)

# --- sub_grade: A1=1 ... G5=35 ---
sg_list = [f"{g}{n}" for g in 'ABCDEFG' for n in range(1, 6)]
sg_map = {sg: i+1 for i, sg in enumerate(sg_list)}
df['sub_grade'] = df['sub_grade'].map(sg_map)

# --- emp_length: ordinal ---
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4,  '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8,  '9 years': 9, '10+ years': 10,
}
df['emp_length'] = df['emp_length'].map(emp_map)  # NaN for missing — imputed later

# --- application_type: binary ---
df['application_type'] = (df['application_type'] == 'Joint App').astype(float)

# --- initial_list_status: binary ---
df['initial_list_status'] = (df['initial_list_status'] == 'f').astype(float)

print("Ordinal encodings done.")
print(df[['term', 'grade', 'sub_grade', 'emp_length']].head(3))

Ordinal encodings done.
   term  grade  sub_grade  emp_length
0  36.0      3         14        10.0
1  36.0      3         11        10.0
2  60.0      2          9        10.0


In [8]:
# --- One-hot encoding ---
ONE_HOT_COLS = ['home_ownership', 'purpose', 'verification_status']

# Include addr_state only if present after missing-value drop
if 'addr_state' in df.columns:
    ONE_HOT_COLS.append('addr_state')

# Fill missing categoricals with 'Unknown' before encoding
for col in ONE_HOT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

df = pd.get_dummies(df, columns=ONE_HOT_COLS, drop_first=True, dtype=float)

print(f"After one-hot encoding: {df.shape[1]} columns")

After one-hot encoding: 133 columns


## 7. Impute Remaining Missing Values

In [9]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f"Columns still with missing values: {len(remaining_missing)}")
print(remaining_missing.sort_values(ascending=False).head(20))

# Median imputation for all remaining numeric columns
num_cols_with_na = df.select_dtypes(include=[np.number]).columns[
    df.select_dtypes(include=[np.number]).isnull().any()
]
medians = df[num_cols_with_na].median()
df[num_cols_with_na] = df[num_cols_with_na].fillna(medians)

print(f"\nAfter imputation — remaining nulls: {df.isnull().sum().sum()}")

Columns still with missing values: 42
mths_since_recent_inq         174071
num_tl_120dpd_2m              117401
mo_sin_old_il_acct            105575
emp_length                     78511
pct_tl_nvr_dlq                 67681
avg_cur_bal                    67549
num_rev_accts                  67528
mo_sin_old_rev_tl_op           67528
mo_sin_rcnt_rev_tl_op          67528
tot_cur_bal                    67527
total_il_high_credit_limit     67527
tot_hi_cred_lim                67527
num_tl_op_past_12m             67527
num_op_rev_tl                  67527
num_rev_tl_bal_gt_0            67527
total_rev_hi_lim               67527
tot_coll_amt                   67527
num_actv_rev_tl                67527
num_il_tl                      67527
num_bc_tl                      67527
dtype: int64

After imputation — remaining nulls: 0


## 8. Final Dataset Summary

In [10]:
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target distribution:\n{df['target'].value_counts()}")
print(f"Default rate: {df['target'].mean():.2%}")
print(f"\nAll dtypes numeric: {(df.dtypes == float).all() or (df.dtypes.isin([float, int, 'int64', 'float64'])).all()}")
print(f"Any remaining nulls: {df.isnull().sum().sum()}")

print("\nFeature list:")
features = [c for c in df.columns if c != 'target']
for f in features:
    print(f"  {f}")

Final shape: 1,345,310 rows x 133 columns
Target distribution:
target
0    1076751
1     268559
Name: count, dtype: int64
Default rate: 19.96%

All dtypes numeric: False
Any remaining nulls: 0

Feature list:
  loan_amnt
  term
  int_rate
  installment
  grade
  sub_grade
  emp_length
  annual_inc
  dti
  delinq_2yrs
  fico_range_low
  inq_last_6mths
  open_acc
  pub_rec
  revol_bal
  revol_util
  total_acc
  initial_list_status
  collections_12_mths_ex_med
  application_type
  acc_now_delinq
  tot_coll_amt
  tot_cur_bal
  total_rev_hi_lim
  acc_open_past_24mths
  avg_cur_bal
  bc_open_to_buy
  bc_util
  chargeoff_within_12_mths
  delinq_amnt
  mo_sin_old_il_acct
  mo_sin_old_rev_tl_op
  mo_sin_rcnt_rev_tl_op
  mo_sin_rcnt_tl
  mort_acc
  mths_since_recent_bc
  mths_since_recent_inq
  num_accts_ever_120_pd
  num_actv_bc_tl
  num_actv_rev_tl
  num_bc_sats
  num_bc_tl
  num_il_tl
  num_op_rev_tl
  num_rev_accts
  num_rev_tl_bal_gt_0
  num_sats
  num_tl_120dpd_2m
  num_tl_30dpd
  num_tl_90

## 9. Save Processed Data
Parquet is much faster to read/write than CSV and preserves dtypes. Falls back to CSV if pyarrow isn't installed.

In [11]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nFile ready for modeling.")

Saved to C:\Users\Administrator\Desktop\DataMiningProject\processed_data.csv
Shape: 1,345,310 rows x 133 columns

File ready for modeling.
